# 01 · Train the ablation (runs A–D)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreyu20sharma/fastdiff-med/blob/main/notebooks/01_train_ablation.ipynb)

**FastDiff-Med** asks whether diffusion sampling can be made fast enough for interactive use and for
producing large synthetic datasets, *without changing the network*. This notebook trains the four
ablation runs. They use the **same class-conditional U-Net, data, optimiser and step budget**. Only
the noise schedule and the prediction target change:

| Run | Noise schedule | Network predicts | Role |
|---|---|---|---|
| A | linear | ε (noise) | baseline DDPM |
| B | cosine | ε | effect of the schedule alone |
| C | linear | v | effect of the target alone |
| D | cosine | v | **Stage 1: FastDiff** (teacher for distillation) |

Each run takes about 25 minutes on a T4 and checkpoints to Drive every 1,000 steps.

### Setup
Mounts Google Drive, clones or updates the repo and installs dependencies. **Everything this notebook
produces is saved to `MyDrive/fastdiff-med/`**, so if the Colab session dies you only rerun this
notebook (training and distillation resume from their last checkpoint).

Set `QUICK = True` for a ~5-minute smoke test with tiny settings. Its outputs go to a separate
`quick/` folder, so they never mix with real results.

*Runtime → Change runtime type → **T4 GPU** before running.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('/content/fastdiff-med'):
    !git clone -q https://github.com/shreyu20sharma/fastdiff-med.git /content/fastdiff-med
%cd /content/fastdiff-med
!git pull -q
!pip install -q -r requirements.txt

QUICK = False   # True = tiny smoke-test settings (~5 min)

ROOT = '/content/drive/MyDrive/fastdiff-med' + ('/quick' if QUICK else '')
CKPT, RESULTS, DOCS = f'{ROOT}/checkpoints', f'{ROOT}/results', f'{ROOT}/docs'
os.environ['MEDMNIST_ROOT'] = '/content/drive/MyDrive/fastdiff-med/data'  # dataset cache (~20 MB)
for d in (CKPT, RESULTS, DOCS, os.environ['MEDMNIST_ROOT']):
    os.makedirs(d, exist_ok=True)

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (switch the runtime to T4!)')
print('Saving to', ROOT)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Image, display
def show(path, width=None):
    if os.path.exists(path):
        display(Image(filename=path, width=width))
    else:
        print('missing:', path)

## 1. The dataset: DermaMNIST
10,015 dermatoscopic images of 7 skin-lesion types (MedMNIST v2), 28×28 RGB, zero-padded to 32×32.
The classes are **heavily imbalanced**: melanocytic nevi are about 67% of the data, while
dermatofibroma and vascular lesions have only about 100 training images each. This imbalance is why
fast synthetic generation is useful (see notebook 03).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from src.data import load_numpy, class_counts, SHORT_NAMES, CLASS_NAMES

imgs, labels = load_numpy('train')
cnt = class_counts(labels)
fig, ax = plt.subplots(figsize=(7, 3), dpi=120)
ax.bar(SHORT_NAMES, cnt, color='#2a78d6')
for i, c in enumerate(cnt):
    ax.annotate(f'{c:,}', (i, c), ha='center', va='bottom', fontsize=8)
ax.set_ylabel('training images'); ax.set_title('Class counts (train split)', loc='left')
ax.spines[['top', 'right']].set_visible(False)
plt.show()

fig, axes = plt.subplots(7, 8, figsize=(8, 7.5), dpi=100)
rng = np.random.default_rng(0)
for k in range(7):
    idx = rng.choice(np.flatnonzero(labels == k), 8, replace=False)
    for j, i in enumerate(idx):
        axes[k, j].imshow(imgs[i]); axes[k, j].axis('off')
    axes[k, 0].set_title(SHORT_NAMES[k], fontsize=8, loc='left')
plt.suptitle('Real samples, one row per class'); plt.tight_layout(); plt.show()
for s, n, c in zip(SHORT_NAMES, CLASS_NAMES, cnt):
    print(f'{s:6s} {c:5d}  {n}')

## 2. Train A, B, C and D
`src/train.py`: AdamW (lr 2e-4, warm-up 500), batch 128, EMA 0.9995, label dropout 0.1 (this
enables classifier-free guidance), random flips and mixed precision. Rerunning a cell **resumes** from
`checkpoints/<run>/last.pt`; a finished run returns immediately.

The log prints `it/s` and an ETA. If you're short on time, train only **A** and **D** (the headline
comparison) and come back for B and C.

In [ ]:
STEPS = 200 if QUICK else 12000
BATCH = 32 if QUICK else 128
RUNS_TO_TRAIN = ['A', 'D', 'B', 'C']     # A and D first: they are the essential pair

for run in RUNS_TO_TRAIN:
    !python -m src.train --run {run} --steps {STEPS} --batch {BATCH} --ckpt_dir {CKPT} --log_every 500

## 3. Training loss
ε- and v-targets live on different scales, so compare the **shape** of the curves (smooth and
converged) rather than their levels. Quality is compared properly with FID in notebook 03.

In [ ]:
from src.evaluate import plot_loss_curves
plot_loss_curves(CKPT, f'{RESULTS}/loss_curves.png')
show(f'{RESULTS}/loss_curves.png')

In [ ]:
# Quick sanity check: 20-step DDIM samples from each trained run (one row per class)
from src.utils import load_model
from src.sampling import sample_images
from src.evaluate import grid_labels
for run in 'ABCD':
    p = f'{CKPT}/{run}/ema.pt'
    if not os.path.exists(p):
        continue
    m, dif, _ = load_model(p)
    x = sample_images(m, dif, grid_labels(8), 'ddim', 20, 2.0, seed=0)
    g = ((x + 1) / 2).clamp(0, 1).view(7, 8, 3, 32, 32).permute(0, 3, 1, 4, 2).reshape(7 * 32, 8 * 32, 3)
    plt.figure(figsize=(4, 3.5), dpi=110); plt.imshow(g.numpy()); plt.axis('off')
    plt.title(f'Run {run} · DDIM 20 steps', fontsize=9, loc='left'); plt.show()